# 03 — Embedding Model Comparison

Benchmark **BGE-Large**, **E5-Large**, and **MiniLM** on medical retrieval.

## Metrics
| Metric | Description |
|--------|-------------|
| **Precision@K** | Fraction of top-K results that are relevant |
| **Recall@K** | Fraction of relevant docs retrieved in top-K |
| **MRR** | Mean Reciprocal Rank — position of first relevant result |
| **Latency** | Average query time (ms) |


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from src.evaluation.evaluator import EmbeddingEvaluator
from src.utils.config import get_settings

cfg = get_settings('../configs/config.yaml')
print('Config loaded ✓')

## 1. Build Retrieval Benchmark

In [ ]:
# Corpus: our built-in medical knowledge docs
from scripts.build_knowledge_base import SAMPLE_DOCUMENTS

corpus = [
    {'id': f'doc_{i}', 'content': d['content']}
    for i, d in enumerate(SAMPLE_DOCUMENTS)
]

queries = [
    {'query': 'low hemoglobin anemia iron deficiency treatment',
     'relevant_doc_ids': ['doc_0', 'doc_6', 'doc_7']},
    {'query': 'diabetes HbA1c blood glucose management metformin',
     'relevant_doc_ids': ['doc_1']},
    {'query': 'LDL HDL cholesterol cardiovascular disease statin',
     'relevant_doc_ids': ['doc_2']},
    {'query': 'thyroid TSH hypothyroidism levothyroxine',
     'relevant_doc_ids': ['doc_3']},
    {'query': 'vitamin D deficiency supplementation bone health',
     'relevant_doc_ids': ['doc_4']},
    {'query': 'kidney disease creatinine GFR renal function',
     'relevant_doc_ids': ['doc_5']},
    {'query': 'complete blood count CBC red blood cells MCV',
     'relevant_doc_ids': ['doc_7']},
]

print(f'Corpus size: {len(corpus)} documents')
print(f'Query set: {len(queries)} queries')

## 2. Run Benchmarks

In [ ]:
# Models to compare — start with MiniLM (fast), add larger ones if GPU available
MODELS = [
    'sentence-transformers/all-MiniLM-L6-v2',    # fastest baseline
    'sentence-transformers/all-mpnet-base-v2',   # strong baseline
    # Uncomment these for full comparison (require more RAM/GPU):
    # 'BAAI/bge-large-en-v1.5',
    # 'intfloat/e5-large-v2',
]

evaluator = EmbeddingEvaluator(corpus, k_values=[1, 3, 5])
results = []

for model in MODELS:
    print(f'\nEvaluating: {model}')
    try:
        m = evaluator.evaluate_model(model, queries)
        results.append(vars(m))
        print(f'  P@1={m.precision_at_1:.3f}  P@3={m.precision_at_3:.3f}  '
              f'P@5={m.precision_at_5:.3f}  MRR={m.mrr:.3f}  '
              f'Latency={m.latency_ms:.1f}ms')
    except Exception as e:
        print(f'  ERROR: {e}')

df = pd.DataFrame(results)
df['model_short'] = df['model_name'].str.split('/').str[-1]
print('\nFull results:')
display(df)

## 3. Visualise Comparison

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Precision@K grouped bar
    x = np.arange(len(df))
    width = 0.25
    axes[0].bar(x - width, df['precision_at_1'], width, label='P@1', color='#2196F3')
    axes[0].bar(x,         df['precision_at_3'], width, label='P@3', color='#4CAF50')
    axes[0].bar(x + width, df['precision_at_5'], width, label='P@5', color='#FF9800')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(df['model_short'], rotation=20, ha='right')
    axes[0].set_ylim(0, 1.1)
    axes[0].set_title('Precision@K')
    axes[0].legend()

    # MRR
    axes[1].bar(df['model_short'], df['mrr'], color='#9C27B0', alpha=0.8)
    for i, v in enumerate(df['mrr']):
        axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
    axes[1].set_ylim(0, 1.1)
    axes[1].set_title('Mean Reciprocal Rank (MRR)')
    axes[1].set_ylabel('MRR')
    plt.setp(axes[1].get_xticklabels(), rotation=20, ha='right')

    # Latency
    axes[2].bar(df['model_short'], df['latency_ms'], color='#F44336', alpha=0.8)
    axes[2].set_title('Query Latency (ms)')
    axes[2].set_ylabel('ms per query')
    plt.setp(axes[2].get_xticklabels(), rotation=20, ha='right')

    plt.suptitle('Embedding Model Comparison — Medical Retrieval', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/embedding_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/embedding_comparison.png')

## 4. Quality vs Speed Trade-off

In [ ]:
if len(df) > 1:
    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(
        df['latency_ms'], df['mrr'],
        s=200, c=df['precision_at_5'], cmap='RdYlGn',
        vmin=0, vmax=1, zorder=5
    )
    plt.colorbar(scatter, ax=ax, label='Precision@5')

    for _, row in df.iterrows():
        ax.annotate(
            row['model_short'],
            (row['latency_ms'], row['mrr']),
            textcoords='offset points', xytext=(8, 4), fontsize=9
        )

    ax.set_xlabel('Latency (ms/query) — lower is better →')
    ax.set_ylabel('MRR — higher is better ↑')
    ax.set_title('Quality vs Speed Trade-off')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../outputs/quality_vs_speed.png', dpi=150)
    plt.show()
else:
    print('Need at least 2 models for trade-off chart. Add more models to MODELS list.')